# Day-to-day evolution of supply and demand
Module for simulating ridesourcing evolution, including pooled rides

Contribution by Arjan de Ruijter - a.j.f.deruijter@tudelft.nl

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys # add MaaSSim and MaaSSim/MaaSSim to path (not needed if already in path)
module_path = os.path.abspath(os.path.join('../..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [2]:
from MaaSSim.utils import save_config, get_config, load_G, generate_demand, initialize_df, empty_series, \
    slice_space, test_space, read_requests_csv
from MaaSSim.maassim import Simulator
from MaaSSim.data_structures import structures as inData
from MaaSSim.d2d_sim import *
from MaaSSim.d2d_demand import *
from MaaSSim.d2d_supply import *
from MaaSSim.d2d_shared import prep_shared_rides
from MaaSSim.decisions import dummy_False

In [3]:
import pandas as pd
import zipfile
import logging
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from matplotlib.lines import Line2D
import numpy as np
import random
import ExMAS
plt.style.use('ggplot')
np.random.seed(0)
random.seed(0)

In [4]:
# Load config
params = get_config('../../data/config/delft.json')  # load configuration
params.paths.albatross = '../../data/albatross'

In [5]:
# Experiment replications and number of threads to be used
params.parallel.nReplications = 1
params.parallel.nThread = 1

# Main experimental settings
params.nP = 500 # travellers
params.nV = 20 # drivers
params.nD = 1 # days
params.simTime = 4 # hours

In [6]:
# Other day-to-day settings
params.evol.drivers.kappa = 0.2 # learning weight (supply-side)
params.evol.drivers.res_wage.mean = 25 #euros/h
params.evol.drivers.gini = 0.35 # gini coefficient used to establish sigma parameter of log-norm distribution of res wage
params.evol.drivers.init_inc_ratio = 1 #expected income of informed drivers at start of sim as ratio of res wage

params.evol.drivers.inform.prob_start = 1 # probability of being informed at start of sim
params.evol.drivers.inform.beta = 0.1 # information transmission rate
params.evol.drivers.inform.std_fact = 0.5 # multiplier of the standard deviation of experienced income used in signal

params.evol.drivers.regist.prob_start = 1 # probability of being registered if informed at start of sim
params.evol.drivers.regist.beta = 0.2 # registration choice model parameter
params.evol.drivers.regist.cost_comp = 20 # daily share of registration costs (euros)
params.evol.drivers.regist.samp = 0.5 # probability of making (de)regist decision
params.evol.drivers.regist.min_work_exp = 0 # Working experience required before deregistration is possible
params.evol.drivers.regist.min_days = 5 # Minimum number of registered days before driver can deregister

params.evol.drivers.particip.beta = 0.1 # participation choice model parameter
params.evol.drivers.particip.probabilistic = True # stochasticity in participation choice

params.evol.travellers.inform.prob_start = 1 # probability that traveller is informed at start of sim
params.evol.travellers.inform.beta = 0.1 # information transmission rate (demand-side)
params.evol.travellers.inform.start_wait = 0 # expected waiting time at start of simulation
params.evol.travellers.inform.std_fact = 0.5 # multiplier of the standard deviation of experienced waiting time used in signal
params.evol.travellers.reject_penalty = 30 * 60 # seconds
params.evol.travellers.kappa = 0.2 # learning weight (demand-side)
params.evol.travellers.min_prob = 0.05 # filtering criterion, when probability is lower when waiting time is zero, never consider RS

params.evol.travellers.mode_pref.mean_vot = 10 # Mean VoT in euro/h
params.evol.travellers.mode_pref.access_multip = 2 # Multiplier of access time compared to in-vehicle time
params.evol.travellers.mode_pref.wait_multip = 2.5 # Multiplier of waiting time compared to in-vehicle time
params.evol.travellers.mode_pref.bike_multip = 2 # Multiplier of biking time compared to in-vehicle time
params.evol.travellers.mode_pref.beta_cost = -0.1592 # util/euro
params.evol.travellers.mode_pref.transfer_pen = 5 * 60 # seconds, to be added to IVT for each transfer
params.evol.travellers.mode_pref.ASC_car = 0 # util, rel to bike
params.evol.travellers.mode_pref.ASC_rs = 0
params.evol.travellers.mode_pref.ASC_pt =  0
params.evol.travellers.mode_pref.ASC_car_sd = 0 # standard deviation in ASCs
params.evol.travellers.mode_pref.ASC_rs_sd = 0
params.evol.travellers.mode_pref.ASC_pt_sd = 0
params.evol.travellers.mode_pref.ASC_bike_sd = 0
params.evol.travellers.mode_pref.gini = params.evol.drivers.gini

# Financial settings
params.platforms.base_fare = 1 #euro
params.platforms.fare = 0 #euro/km
params.platforms.min_fare = 0 # euro
params.platforms.comm_rate = 0.05 #rate
params.drivers.fuel_costs = 0.25 #euro/km

# Properties alternative modes
params.alt_modes.pt.option = False
params.alt_modes.pt.base_fare = 0.99 # euro
params.alt_modes.pt.km_fare = 0.174 # euro/km
params.alt_modes.car.km_cost = 0.5 # euro/km
params.alt_modes.car.diff_parking = True # different parking tariffs in city
params.alt_modes.car.park_cost = 7.5 # euro
params.alt_modes.car.park_cost_center = 15 # euro
params.alt_modes.car.access_time = 10 * 60 # s
params.speeds.bike = (1/2.5) * params.speeds.ride # m/s

# Regulation
params.platforms.reg_cap = np.inf # registration cap
params.platforms.ptcp_cap = np.inf # daily participation cap

# Demand settings
# params.demand_structure.origins_dispertion = -0.0003
# params.demand_structure.destinations_dispertion = -0.0003
params.dist_threshold_min = 2000 # min dist
# params.dist_threshold = 100000 # max dist

# Start time
# params.t0 = pd.Timestamp.now()
params.t0 = pd.Timestamp(2021, 11, 1, 9)

In [7]:
# Pooling settings
params.shareability.offered = True
params.shareability.avg_speed = params.speeds.ride
params.shareability.min_discount = 0.5
params.shareability.add_discount = 0.0
params.shareability.shared_discount = params.shareability.min_discount
params.shareability.delay_value = 1
params.shareability.WtS = 1.1 # Willingness to share
params.shareability.price = 1.5 #eur/km
params.shareability.VoT = 0.0035 #eur/s
params.shareability.matching_obj = 'u_pax' #minimize VHT for vehicles
params.shareability.pax_delay = 0
params.shareability.horizon = 600
params.shareability.max_degree = 2
params.shareability.nP = params.nP
params.shareability.share = 1
params.shareability.without_matching = True

In [8]:
inData = load_G(inData, params, stats=True, set_t=False)  # download graph for the 'params.city' and calc the skim matrices
if params.alt_modes.car.diff_parking:
    inData = diff_parking(inData) # determine which nodes are in center

In [9]:
inData = generate_demand(inData, params, avg_speed = True)

In [10]:
# Load processed Albatross file, the OTP result, and compute PT fares
# inData = load_albatross_proc(inData, params, avg_speed = True)
# inData.requests = inData.requests.drop(['orig_geo', 'dest_geo', 'origin_y', 'origin_x', 'destination_y', 'destination_x', 'time'], axis = 1)
# inData.pt_itinerary = load_OTP_result(params)
# inData = consist_OTP_alba(inData, params)

In [11]:
# Prepare supply and demand attributes
inData.passengers = prefs_travs(inData, params)
all_pax = mode_filter(inData, params)
inData.passengers = all_pax[all_pax.mode_choice == "day-to-day"]
inData.requests = inData.requests[inData.requests.pax_id.isin(inData.passengers.index)]
if params.alt_modes.pt.option:
    inData.pt_itinerary = inData.pt_itinerary[inData.pt_itinerary.pax_id.isin(inData.passengers.index)]
    inData.passengers.reset_index(drop=True, inplace=True)
    inData.requests.reset_index(drop=True, inplace=True)
    inData.pt_itinerary.reset_index(drop=True, inplace=True)
inData.requests['pax_id'] = inData.requests.index
inData.pt_itinerary['pax_id'] = inData.pt_itinerary.index
inData.passengers['informed'] = np.random.rand(len(inData.passengers)) < params.evol.travellers.inform.prob_start
inData.passengers['expected_wait'] = params.evol.travellers.inform.start_wait
inData.passengers['expected_wait_pool'] = params.evol.travellers.inform.start_wait
inData.passengers['expected_pool_discount'] = params.shareability.min_discount
inData.passengers['expected_pool_delay'] = 0
fixed_supply = generate_vehicles_d2d(inData, params)
inData.vehicles = fixed_supply.copy()
inData.vehicles.platform = inData.vehicles.apply(lambda x: 0, axis = 1)
inData.passengers.platforms = inData.passengers.apply(lambda x: [0], axis = 1)
inData.requests['platform'] = inData.requests.apply(lambda row: inData.passengers.loc[row.name].platforms[0], axis = 1) 
inData.platforms = pd.concat([inData.platforms,pd.DataFrame(columns=['base_fare','comm_rate','min_fare'])])
inData.platforms = initialize_df(inData.platforms)
inData.platforms.loc[0]=[params.platforms.fare,'Uber',30,params.platforms.base_fare,params.platforms.comm_rate,params.platforms.min_fare,]

In [12]:
inData = ExMAS.main(inData, params.shareability, plot=False) # create shareability graph (ExMAS) 

28-02-23 14:21:52-INFO-Initializing pairwise trip shareability between 500 and 500 trips.
28-02-23 14:21:52-INFO-creating combinations
28-02-23 14:21:53-INFO-249500	 nR*(nR-1)
28-02-23 14:21:57-CRITICAL-LIFO pairs assertion failed
28-02-23 14:21:57-WARNING-Empty DataFrame
Columns: [origin_i, destination_i, ttrav_i, treq_i, delta_i, dist_i, VoT_i, origin_j, destination_j, ttrav_j, treq_j, delta_j, dist_j, VoT_j, i, j, t_oo, delay, delay_i, delay_j, t_od, t_dd, ttrav, kind, indexes, indexes_orig, indexes_dest, u_i, u_j, t_i, t_j, delta_ij, delta_ji, delta, u_pax]
Index: []

[0 rows x 35 columns]
28-02-23 14:21:57-WARNING-Empty DataFrame
Columns: [origin_i, destination_i, ttrav_i, treq_i, delta_i, dist_i, VoT_i, origin_j, destination_j, ttrav_j, treq_j, delta_j, dist_j, VoT_j, i, j, t_oo, delay, delay_i, delay_j, t_od, t_dd, ttrav, kind, indexes, indexes_orig, indexes_dest, u_i, u_j, t_i, t_j, delta_ij, delta_ji, delta, u_pax]
Index: []

[0 rows x 35 columns]
28-02-23 14:21:57-WARNING-Emp

In [15]:
# Day-to-day simulation (incl. processing)
sim = Simulator(inData, params=params,
                    kpi_veh = D2D_veh_exp,
                    kpi_pax = d2d_kpi_pax,
                    f_driver_out = D2D_driver_out,
                    f_trav_out = d2d_no_request,
                    f_trav_mode = dummy_False,
                    logger_level=logging.WARNING)  # initialize

evol_micro = init_d2d_dotmap()
for day in range(params.get('nD', 1)):  # run iterations
    inData.passengers = mode_preday(inData, params)
    temp_rides = inData.sblts.rides.copy()
    temp_reqs = inData.sblts.requests.copy()
    
    rs_users = inData.passengers[(inData.passengers.mode_day == 'rs') | (inData.passengers.mode_day == 'pool')].index.tolist()
    poolers = inData.passengers[inData.passengers.mode_day == 'pool'].index.tolist()
    inData.sblts.rides = inData.sblts.rides[inData.sblts.rides.apply(lambda x: all(i in rs_users for i in x.indexes), axis=1)] # filter out all travellers opting for mode outside ride-hailing market
    inData.sblts.rides = inData.sblts.rides[inData.sblts.rides.apply(lambda x: (all(i in poolers for i in x.indexes) or x.kind == 1), axis=1)]  # filter out pooled trips for individuals opting for private ride
    inData.sblts.requests = inData.sblts.requests[inData.sblts.requests.apply(lambda x: x.pax_id in rs_users, axis=1)]
    abc = inData.sblts.rides.copy()
    defg = inData.sblts.requests.copy()
#     inData.sblts.requests = inData.sblts.requests[inData.sblts.requests.apply(lambda x: x.index in poolers, axis=1)]
    
    inData = prep_shared_rides(inData, params.shareability)  # prepare schedules
    sim.make_and_run(run_id=day)  # prepare and SIM
    sim.output()  # calc results
    sim.last_res = sim.res[day].copy()
    del sim.res[day]
    hij = inData.sblts.rides.copy()
    klm = inData.sblts.requests.copy()

    drivers_summary = update_d2d_drivers(sim=sim,params=params)
    travs_summary = update_d2d_travellers(sim=sim,params=params)
    
    exp_df = update_work_exp(inData, drivers_summary)
    inData.vehicles.work_exp = exp_df.work_exp
    inData.days_since_reg = exp_df.days_since_reg

    res_inf_driver = wom_driver(inData, params = params)
    inData.vehicles.informed = res_inf_driver
    inData.vehicles.expected_income = learning_unregist(inData, drivers_summary, params = params)
    
    res_regist = platform_regist(inData, drivers_summary, params = params)
    inData.vehicles.registered = res_regist.registered
    inData.vehicles.work_exp = res_regist.work_exp
    inData.vehicles.pos = fixed_supply.pos
    inData.vehicles.rejected_reg = res_regist.rejected_reg
    exp_inf_trav = travs_summary.loc[travs_summary.informed]
    average_xp_wait = exp_inf_trav.corr_xp_wait.mean() / 60
    res_inf_trav = wom_trav(inData, travs_summary, params = params)
    inData.passengers.informed = res_inf_trav.informed
    inData.passengers.expected_wait = res_inf_trav.perc_wait
    
    inData.sblts.rides = temp_rides.copy()
    inData.sblts.requests = temp_reqs.copy()
    
    evol_micro = d2d_summary_day(evol_micro, drivers_summary, travs_summary, day)

28-02-23 14:24:08-WARNING-Setting up 4h simulation at 2021-11-01 07:00:18 for 20 vehicles and 500 passengers in Delft, Netherlands
Chosen req: 0
Chosen req: 1
requests, id: 2, time: 27
Chosen req: 3
schedule triggered, id: 2, time: 72
requests, id: 4, time: 90
Chosen req: 5
Chosen req: 9
Chosen req: 10
Chosen req: 11
Chosen req: 12
schedule triggered, id: 4, time: 352
Chosen req: 16
requests, id: 21, time: 456
schedule triggered, id: 21, time: 456
Chosen req: 19
requests, id: 29, time: 719
schedule triggered, id: 29, time: 719
Chosen req: 20
requests, id: 32, time: 808
Chosen req: 14
Chosen req: 30
requests, id: 34, time: 910
schedule triggered, id: 34, time: 910
Chosen req: 15
Chosen req: 28
Chosen req: 26
schedule triggered, id: 32, time: 1016
Chosen req: 33
Chosen req: 41
Chosen req: 45
Chosen req: 48
requests, id: 52, time: 1414
schedule triggered, id: 52, time: 1414
Chosen req: 46
requests, id: 58, time: 1617
Chosen req: 51
Chosen req: 47
schedule triggered, id: 58, time: 1644
Cho

In [16]:
evol_micro, evol_agg = d2d_agg_statistics(evol_micro, params) # multi-day stats

In [17]:
# Save d2d stats to zip file
with zipfile.ZipFile('evol.zip', 'w') as csv_zip:
    csv_zip.writestr("evol_agg_supply.csv", evol_agg.supply.to_csv())
    csv_zip.writestr("evol_agg_demand.csv", evol_agg.demand.to_csv())

In [18]:
evol_agg.supply

,inform,regist,rejected_reg,particip,reject_particip,mean_perc_inc,mean_perc_inc_ptcp,mean_perc_inc_reg,mean_exp_inc
day,,,,,,,,,
0,20,20,0,8,0,89.227317,77.47586,89.227317,0.29125


In [19]:
evol_agg.demand

,inform,requests,req_solo,req_pool,gets_offer_solo,gets_offer_pooling,mean_wait_solo,corr_mean_wait_solo,mean_wait_pooling,corr_mean_wait_pooling,perc_wait_solo,perc_wait_req,bike,car,pt
day,,,,,,,,,,,,,,,
0,500,348,152,196,96,143,1312.543933,1465.224138,1157.548117,1069.12069,212.6896,204.047126,127,25,0


In [ ]:
# Plot number of drivers and income
fig, axes = plt.subplots(nrows=5, ncols=1, figsize = (8,12.5), sharex = True)
evol_agg.supply[['inform','regist','particip']].plot(ax = axes[0], color=['lightsteelblue','tab:blue','midnightblue'])
axes[0].set_title('(A) Ridesourcing supply')
axes[0].legend(['Informed','Registered','Participating'])
axes[0].set_ylim([0,params.nV + 25])
axes[0].set_ylabel('Number of drivers')
evol_agg.supply[['mean_perc_inc','mean_exp_inc']].plot(ax = axes[2], color=['lightsteelblue','midnightblue'])
axes[2].set_title('(C) Driver earnings')
axes[2].legend(['Expected','Experienced'])
axes[2].set_ylim([0,math.ceil(max(evol_agg.supply.mean_perc_inc.max(),evol_agg.supply.mean_exp_inc.max())/50)*50])
axes[2].set_ylabel('Income (\u20ac)')


evol_agg.demand[['requests','bike','car','pt']].plot.area(ax = axes[1])
h,l = axes[1].get_legend_handles_labels()
evol_agg.demand['inform'].plot(ax = axes[1], color = 'black', linestyle = 'dashed', label = 'informed')
line = Line2D([0], [0],color='black', linestyle ='dashed')
axes[1].set_title('(B) Demand')
axes[1].set_ylim([0,len(inData.passengers) * 1.1])
axes[1].set_ylabel('Number of travellers')
h.extend([line])
axes[1].legend(labels=["Ridesourcing","Bike","Car","Public transport","Informed"], handles=h)

ax_sec = axes[3].twinx()
evol_agg.demand['proport_match'] = evol_agg.demand.gets_offer / evol_agg.demand.requests * 100
evol_agg.demand['mean_wait'].apply(lambda x: 1/60 * x).plot(ax = axes[3], label='Experienced waiting time', color ='lightsalmon')
evol_agg.demand['corr_mean_wait'].apply(lambda x: 1/60 * x).plot(ax = axes[3], label='Corrected exp. waiting time', color ='rosybrown')
evol_agg.demand['perc_wait'].apply(lambda x: 1/60 * x).plot(ax = axes[3], label='Expected waiting time', color ='tomato')
evol_agg.demand['proport_match'].plot(ax = ax_sec, color = 'grey', label='Share of requests returned with offer', linestyle = 'dotted')
lines_1, labels_1 = axes[3].get_legend_handles_labels()
lines_2, labels_2 = ax_sec.get_legend_handles_labels()
lines = lines_1 + lines_2
labels = labels_1 + labels_2
axes[3].legend(lines, labels, loc=0)
axes[3].set_title('(D) Level of service')
axes[3].set_ylabel('Mean wait. time (min)')
ax_sec.set_ylabel('Request success')
ax_sec.yaxis.set_major_formatter(mtick.PercentFormatter())
max_val = max(evol_agg.demand.mean_wait.max(),evol_agg.demand.corr_mean_wait.max(),evol_agg.demand.perc_wait.max())
axes[3].set_ylim([0,math.ceil((1/60)*max_val/2)*2+0.5])
ax_sec.set_ylim([0,100+5])
ax_sec.grid(None)

proport_rs = evol_agg.demand.requests / evol_agg.demand.inform * 100 
proport_rs.plot(ax = axes[4], label = 'Travellers - Mode share of ridesourcing', color='maroon')
axes[4].set_title('(E) Platform utilisation')
axes[4].set_ylim([0,100+2])
axes[4].set_ylabel('Platform utilisation')

axes[4].yaxis.set_major_formatter(mtick.PercentFormatter())

proport_work = evol_agg.supply.particip / evol_agg.supply.regist * 100
proport_work.plot(ax = axes[4], label = 'Drivers - Labour participation rate', color ='midnightblue')
lines, labels = axes[4].get_legend_handles_labels()
axes[4].legend(labels)

plt.savefig('d2d-evo.png')

---

In [ ]:
inData.keys()

In [ ]:
# inData.pt_itinerary.to_csv('inData_pt-itinerary.csv')
inData.vehicles

In [ ]:
inData.stats.center

In [ ]:
evol_micro.supply.inform

In [ ]:
inData.skim.iloc[:20, :20].to_csv('inData_skim.csv')

In [ ]:
(evol_micro.demand.gets_offer * evol_micro.demand.requests).sum()

In [ ]:
inData.passengers.to_csv('inData_passengers.csv')

In [ ]:
evol_micro.demand.gets_offer

In [ ]:
inData.vehicles

In [ ]:
# sim.last_res.pax_exp.head(50).to_csv('pax_exp.csv')
sim.last_res.keys()

In [ ]:
# sim.last_res.veh_kpi.head(50).to_csv('veh_kpi.csv')
sim.last_res.veh_exp

In [ ]:
sim.last_res.pax_exp

In [ ]:
import networkx as nx
df = nx.to_pandas_edgelist(inData.G)
df.to_csv('edges.csv')

In [ ]:
inData.pt_itinerary

In [ ]:
abc

In [ ]:
defg

In [ ]:
inData.keys()

In [ ]:
inData.sblts.keys()

In [ ]:
inData.sblts.requests

In [ ]:
inData.the_skim.sort_index().sort_index(axis=1).head(20)

In [ ]:
inData.sblts.requests

In [ ]:
inData.sblts.R[2]

In [ ]:
inData.sblts.rides

In [ ]:
inData.sblts.requests

In [ ]:
sim.runs[0]['trips'][sim.runs[0]['trips']['pax'] == 1722]

In [ ]:
inData.requests[inData.requests['pax_id'] == 1722]['sim_schedule']

In [ ]:
inData.requests.loc[439].sim_schedule

In [ ]:
# inData.requests.loc[421]
inData.requests.loc[13].sim_schedule

In [ ]:
inData.keys()

In [ ]:
inData.sblts.keys()

In [ ]:
inData.sblts.rides[inData.sblts.rides.selected == 1].tail(110)

In [ ]:
inData.sblts.requests.loc[21]

In [ ]:
sim.runs[1].rides

In [ ]:
sim.runs[0].rides[sim.runs[0].rides['event']=='DEPARTS_FROM_PICKUP'].head(50)

In [ ]:
# sim.runs[0].rides[sim.runs[0].rides['t']>=4000].head(50)
sim.runs[0].rides.head(50)

In [ ]:
sim.runs[0].trips[sim.runs[0].trips['pax']==19]

In [ ]:
inData.sblts.schedule['sim_schedule']

In [ ]:
inData.requests['sim_schedule'].head(50)

In [ ]:
inData.requests.sim_schedule.loc[20]

In [ ]:
inData.keys()

In [ ]:
inData.vehicles

In [ ]:
inData.passengers

In [ ]:
sim.runs[0].rides.head(50)

In [ ]:
sim.runs[0].trips[sim.runs[0].trips['pax']==257]

In [ ]:
inData.keys()

In [ ]:
inData.sblts.keys()

In [ ]:
inData.sblts.requests.head(50)

In [ ]:
inData.sblts.schedule.head(70)

In [ ]:
inData.requests

In [ ]:
inData.requests.loc[407].sim_schedule

In [ ]:
inData.sblts.SINGLES

In [ ]:
inData.passengers.loc[201]

In [ ]:
inData.sblts.requests.loc[201]

In [ ]:
inData.sblts.rides

In [ ]:
inData.passengers[inData.passengers.mode_day == 'pool'].index.tolist()

In [ ]:
# inData.sblts.rides[inData.sblts.rides.apply(lambda x: any(i in pool for i in x.indexes))]
pool=[2,5]
inData.sblts.rides.apply(lambda x: any(i in pool for i in x.indexes), axis=1)

In [ ]:
inData.passengers.loc[422]

In [ ]:
inData.sblts.requests

In [ ]:
inData.sblts.schedule.head(50)

In [ ]:
travs_summary

In [ ]:
evol_micro.demand.wait_time

In [ ]:
evol_micro.demand.corr_wait_time

In [ ]:
evol_micro.demand.req_pool

In [ ]:
evol_micro.demand.perc_wait

In [ ]:
sim.runs[2].trips.head(50)

In [ ]:
inData.sblts.rides.head(90)

In [ ]:
inData.sblts.requests.loc[453]

In [ ]:
xyz = inData.sblts.rides[inData.sblts.rides.apply(lambda x: all(i in rs_users for i in x.indexes), axis=1)] # filter out all travellers opting for mode outside ride-hailing market
xyz = xyz[xyz.apply(lambda x: (all(i in poolers for i in x.indexes) or x.kind == 1), axis=1)]  # filter out pooled trips for individuals opting for private ride

xyz
# poolers

In [35]:
inData.sblts.schedule.tail(50)

,indexes,u_pax,u_veh,kind,u_paxes,times,indexes_orig,indexes_dest,index,lambda_r,PassHourTrav_ns,row,selected,degree,nodes,req_id,sim_schedule
6611,"[444, 447]",6.65650,250,21,"[3.2359, 3.4206000000000003]","[12916.0, 3, 42, 205]","[444, 447]","[447, 444]",6611,-2.521127,71,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1,2,"[None, 672526443, 6248301254, 44844070, 126911...","[None, 444, 447, 447, 444]",node time req_id od 0 ...
6643,"[426, 438]",7.17820,442,21,"[4.47585, 2.70235]","[12589.0, 274, 32, 136]","[426, 438]","[438, 426]",6643,-5.696970,66,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1,2,"[None, 44739628, 1474883280, 44859610, 1383461...","[None, 426, 438, 438, 426]",node time req_id od 0 ...
6673,"[74, 76]",6.25795,247,21,"[3.4302, 2.82775]","[2190.0, 61, 35, 151]","[74, 76]","[76, 74]",6673,-2.686567,67,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1,2,"[None, 44877771, 44891423, 4124549374, 1391531...","[None, 74, 76, 76, 74]",node time req_id od 0 ...
6731,"[363, 379]",7.66780,489,21,"[5.0988, 2.569]","[10209.0, 253, 31, 205]","[363, 379]","[379, 363]",6731,-5.791667,72,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1,2,"[None, 2670677182, 7399789770, 1830076520, 139...","[None, 363, 379, 379, 363]",node time req_id od 0 ...
6738,"[211, 218]",6.10605,451,21,"[4.013375, 2.092675]","[6244.5, 141, 24, 286]","[211, 218]","[218, 211]",6738,-7.673077,52,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1,2,"[None, 44837765, 1435362502, 1552651408, 14027...","[None, 211, 218, 218, 211]",node time req_id od 0 ...
6775,"[318, 324]",6.94240,216,21,"[3.1380749999999997, 3.8043250000000004]","[9169.5, 108, 46, 62]","[318, 324]","[324, 318]",6775,-1.918919,74,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1,2,"[None, 44840841, 2323108138, 1433962926, 14027...","[None, 318, 324, 324, 318]",node time req_id od 0 ...
6830,"[180, 197]",8.28950,441,21,"[4.937950000000001, 3.35155]","[5373.0, 364, 42, 35]","[180, 197]","[197, 180]",6830,-4.188235,85,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1,2,"[None, 44753493, 44850245, 44830592, 1410536895]","[None, 180, 197, 197, 180]",node time req_id od 0 ...
6955,"[224, 226]",9.61405,605,21,"[6.7637, 2.85035]","[6427.0, 286, 29, 290]","[224, 226]","[226, 224]",6955,-6.469136,81,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1,2,"[None, 582000012, 44814387, 44795688, 1433962853]","[None, 224, 226, 226, 224]",node time req_id od 0 ...
7231,"[70, 82]",8.11155,335,21,"[5.5146500000000005, 2.5969]","[2227.0, 61, 25, 249]","[70, 82]","[82, 70]",7231,-3.589041,73,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1,2,"[None, 527378782, 2323108159, 1552650472, 1448...","[None, 70, 82, 82, 70]",node time req_id od 0 ...
7254,"[263, 278]",7.63965,485,21,"[5.561875000000001, 2.077775]","[7835.5, 370, 24, 91]","[263, 278]","[278, 263]",7254,-5.830986,71,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1,2,"[None, 1519889961, 44814387, 44851133, 1448535...","[None, 263, 278, 278, 263]",node time req_id od 0 ...


In [41]:
inData.requests.loc[489]

pax_id                                                         489
origin                                                  1402734973
destination                                             1436427064
treq                                           2021-11-01 10:54:36
tdep                                                           NaN
ttrav                                              0 days 00:05:40
tarr                                           2021-11-01 11:00:16
tdrop                                                          NaN
shareable                                                     True
schedule_id                                                    NaN
dist                                                          3404
car_park_cost                                                  7.5
dest_center                                                  False
platform                                                         0
ride_id                                                       

In [ ]:
inData.requests.loc[256].sim_schedule

In [ ]:
inData.passengers.loc[256]

In [ ]:
inData.sblts.rides

In [ ]:
abc

In [ ]:
inData.sblts.schedule.loc[9084].sim_schedule

In [ ]:
inData.sblts.schedule.tail(50)

In [ ]:
inData.passengers.loc[256]

In [ ]:
hij

In [ ]:
klm

In [36]:
sim.runs[0].trips[sim.runs[0].trips.pax == 473]

,pax,pos,t,event,veh_id
0,473,1830058806,0,STARTS_DAY,NaN
1,473,1830058806,13761,REQUESTS_RIDE,NaN
2,473,1830058806,13977,RECEIVES_OFFER,NaN
3,473,1830058806,13977,ACCEPTS_OFFER,NaN
4,473,1830058806,13997,ARRIVES_AT_PICKUP,20.0
5,473,1830058806,14182,MEETS_DRIVER_AT_PICKUP,20.0
6,473,1830058806,14212,DEPARTS_FROM_PICKUP,20.0
7,473,4486831034,15160,ARRIVES_AT_DROPOFF,20.0
8,473,4486831034,15170,SETS_OFF_FOR_DEST,NaN
9,473,4486831034,15170,ARRIVES_AT_DEST,NaN


In [ ]:
sim.runs[0].trips[sim.runs[0].trips.pax == 193]

In [ ]:
sim.runs[2].rides[sim.runs[2].rides.veh == 6].tail(100)

In [ ]:
sim.runs[2].rides[sim.runs[2].rides.apply(lambda x: x.paxes == [119], axis=1)]

In [ ]:
inData.sblts.schedule.loc[9057].sim_schedule

In [ ]:
klm.loc[119]

In [ ]:
hij.loc[9084].

In [ ]:
inData.sblts.schedule.loc[9084]

In [ ]:
koekoek = inData.requests.copy()
koekoek['hoi'] = inData.sblts.requests.shareable
koekoek

In [ ]:
inData.sblts.requests.loc[259]

In [ ]:
koekoek.hoi.fillna(False)

In [ ]:
travs_summary.loc[463]

In [ ]:
sim.last_res.pax_exp.loc[463]

In [ ]:
(inData.requests.loc[378].treq - params.t0).seconds

In [ ]:
evol_micro.supply

In [ ]:
sim.runs[0].outcomes

In [23]:
abc = travs_summary[travs_summary.chosen_mode == 'pool']
abc[abc.gets_offer]

,orig,dest,t_req,tt_min,dist,informed,requests,gets_offer,accepts_offer,xp_wait,xp_ivt,xp_ops,act_shared,xp_discount,xp_tt_total,init_perc_wait,corr_xp_wait,new_perc_wait,chosen_mode
pax,,,,,,,,,,,,,,,,,,,
2,4053473202,1385072823,2021-11-01 07:00:45,0 days 00:04:01,2416,True,True,True,True,496.0,241.0,55.0,9999,1.0,792.0,4.6,496.0,102.88,pool
3,44862225,44759522,2021-11-01 07:01:15,0 days 00:06:38,3983,True,True,True,True,211.0,585.0,55.0,9999,1.0,851.0,0.0,211.0,42.20,pool
4,1679761151,44839603,2021-11-01 07:01:48,0 days 00:03:32,2125,True,True,True,True,755.0,212.0,55.0,9999,1.0,1022.0,21.8,755.0,168.44,pool
5,2381851915,44841824,2021-11-01 07:02:03,0 days 00:06:26,3866,True,True,True,True,59.0,777.0,55.0,9999,1.0,891.0,1.2,59.0,12.76,pool
11,1552650415,44734726,2021-11-01 07:05:54,0 days 00:06:10,3703,True,True,True,True,304.0,465.0,55.0,9999,1.0,824.0,39.6,304.0,92.48,pool
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
476,4244308801,1552650539,2021-11-01 10:51:17,0 days 00:04:27,2678,True,True,True,True,353.0,665.0,40.0,9999,1.0,1058.0,110.0,353.0,158.60,pool
478,1435362418,27082148,2021-11-01 10:52:06,0 days 00:07:05,4253,True,True,True,True,266.0,914.0,40.0,9999,1.0,1220.0,27.0,266.0,74.80,pool
486,1569647891,44845664,2021-11-01 10:54:06,0 days 00:04:58,2980,True,True,True,True,355.0,298.0,40.0,9999,1.0,693.0,0.0,355.0,71.00,pool


In [31]:
travs_summary.loc[489]

orig                       1402734973
dest                       1436427064
t_req             2021-11-01 10:54:36
tt_min                0 days 00:05:40
dist                             3404
informed                         True
requests                         True
gets_offer                       True
accepts_offer                    True
xp_wait                         14566
xp_ivt                            340
xp_ops                             40
act_shared                       9999
xp_discount                         1
xp_tt_total                     14946
init_perc_wait                   2698
corr_xp_wait                    14566
new_perc_wait                  5071.6
chosen_mode                      pool
Name: 489, dtype: object